# Flatand Graph Data Import Pipeline

## Approach Sketch
To be implemented...
- one node per outgoing edge, label with where it comes from
- look at switches: at fixed distance which 2 are closest -> suggest using GeoPandas API for this.
- add actions as edge attributes (LFR disjointly to all outgoing)
- export enhanced osmnx 
- write importer for enhanced osmnx to GraphRailEnv -> describe what we expect as input, which attributes etc.
- timetable for graph env
- unit test with policy reaching target (possibly hard-coded actions)
- snaphost-visualization (preferrably one osmnx export with railway features -> could be a moving house or roundabout if there is no train feature...)


## Documentation
### OSMnx and GeoPandas API documentation
* https://osmnx.readthedocs.io/en/stable/user-reference.html
* https://github.com/gboeing/osmnx-examples
* https://geopandas.org/en/stable/docs/user_guide.html

### OSM Railway documentation
* https://wiki.openstreetmap.org/wiki/Railways
* https://wiki.openstreetmap.org/wiki/Railway_stations
* https://wiki.openstreetmap.org/wiki/Tag:railway=signal
* https://wiki.openstreetmap.org/wiki/Tag:railway=switch
* https://wiki.openstreetmap.org/wiki/Tag:railway=rail
* https://wiki.openstreetmap.org/wiki/Tag:route%3Drailway


### OpenRailwayMap -> inspect Railway features
* https://openrailwaymap.app/#view=18.86/47.1194931/9.3132586&style=signals

### Insepct OSM by Node ID / Way ID
* https://www.openstreetmap.org/node/2048248546/history/4
* https://www.openstreetmap.org/way/194299504

In [ ]:
# !python -m pip install jupyter osmnx matplotlib  "folium>=0.12" matplotlib mapclassify
import osmnx as ox

ox.settings.log_console = True
ox.__version__

In [ ]:
# ox.settings.overpass_url='http://localhost:12347'
# ox.settings.overpass_rate_limit=False

In [ ]:
place = "Walenstadt"
# tiles = "cartodbpositron"
mk = {"radius": 6}
# cols = ["height", "addr:housenumber", "addr:street", "addr:postcode"]

## Download OSMnx graph

In [ ]:
ox.settings.all_oneway = True
G = ox.graph.graph_from_place(
   place,
   retain_all=False, # If True, return the entire graph even if it is not connected. If False, retain only the largest weakly connected component.
   truncate_by_edge=True, # If True, retain nodes the outside bounding box if at least one of the node’s neighbors lies within the bounding box.
   simplify=True, # If True, simplify graph topology with the simplify_graph function.
   custom_filter='["railway"~"rail"]',
)
fig, ax = ox.plot.plot_graph(G, node_size=0, edge_color="w", edge_linewidth=0.2)

In [ ]:
gdf = ox.features.features_from_place(place, 
                                      tags={"building": True}
                                     )
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)

In [ ]:
edges.explore(m=m, color="purple")

In [ ]:
nodes.explore(marker_kwds={"radius": 8})

In [ ]:
list(G.nodes(data=True))

## Visualize networkx graph

In [ ]:
import networkx as nx
nx.draw(G, labels={n[0]: str(n[0]) for n in list(G.nodes(data=True))})

## Stations

In [ ]:
stations = ox.features.features_from_place(place, tags={"railway": "station"})
stations.explore(m=m, 
                 #tiles=tiles, 
                 color="yellow", tooltip="name", marker_kwds=mk)

## Signals

In [ ]:
signals = ox.features.features_from_place(place, tags={"railway": "signal"})

In [ ]:
m = gdf.explore(
    #tiles=tiles,
    #tooltip=cols
)
nodes,edges = ox.convert.graph_to_gdfs(G)
nodes.explore(m=m, color="purple")

## Export graph to graphml

In [ ]:
# TODO must not be simplified - alternative?
ox.io.save_graphml(G, filepath="walensee.graphml")

## Import graph from graphml

In [ ]:
GG = ox.io.load_graphml(filepath="walensee.graphml")

In [ ]:
# TODO MultiDiGraph - due to non-simplify?
GG = nx.DiGraph(GG)

## Graph Rail Env

In [ ]:
for n, d in GG.nodes(data=True):
    if "railway" not in d or d["railway"] != "switch":
        continue
    print(type(n))
    print(n,d)

In [ ]:
SWITCH=2042222936

In [ ]:
for n in list(GG.successors(SWITCH)):
    print(n, GG.get_edge_data(SWITCH, n, default=None))

In [ ]:
TARGET=2042193124

In [ ]:
for n in list(GG.predecessors(SWITCH)):
    print(n, GG.get_edge_data(n, SWITCH, default=None))

In [ ]:
#!python -m pip install flatland-rl==4.3.0
from flatland.core.env_observation_builder import DummyObservationBuilder
from flatland.envs.graph.rail_graph_transition_map import GraphTransitionMap
from flatland.core.graph.graph_resource_map import GraphResourceMap
from flatland.envs.graph.distance_map import GraphDistanceMap
from flatland.envs.graph_rail_env import GraphRailEnv
from flatland.envs.timetable_utils import Timetable
from flatland.envs import malfunction_generators as mal_gen
from flatland.envs.rewards import DefaultRewards
from flatland.envs.rail_env import RailEnvActions

In [ ]:
# add actions
for v1,v2, edge_data in GG.edges(data=True):
    # TODO actions on outgoing edge data plus v1's prohibited_actions attribute should partition all actions 
    edge_data["actions"] = {RailEnvActions.MOVE_FORWARD}
    # TODO do we still need it?
    edge_data["straight"] = v2
    #GG.edges(v1,v2)["actions"]= {RailEnvActions.MOVE_FORWARD}
    # print(v1,v2,edge_data)

In [ ]:
resource_map = {}
for n1, n2 in GG.edges:
    resource_map[n1] = str(sorted([n1,n2]))
# print(resource_map)

agent_waypoints = {
    0: [[SWITCH], [TARGET]]
}
timetable = Timetable(
    earliest_departures=[[0,None]],
    latest_arrivals=[[None,5]],
    max_episode_steps=99,
)

agent_speeds = {
    0: 1/2
}

graph_env = GraphRailEnv.from_graph(
    g=GG,
    resource_map=resource_map,
    agent_waypoints=agent_waypoints,
    agent_speeds=agent_speeds,
    observation_builder=DummyObservationBuilder(),
    malfunction_generator=mal_gen.NoMalfunctionGen(),
    timetable_generator=lambda *args, **kwargs: timetable,
    seed=42,
    rewards=DefaultRewards(),
)
graph_env.reset()

In [ ]:
GG.get_edge_data(SWITCH,TARGET)

In [ ]:
graph_env.rail.g.get_edge_data(SWITCH,TARGET)

In [ ]:
graph_env.reset()
for _ in range(99):
    print("==============================")
    print(f"elapsed before step {graph_env._elapsed_steps}")
    for agent in graph_env.agents:
        print(f"agent.initial_configuration={agent.initial_configuration}")
        print(f"agent.current_configuration={agent.current_configuration}")
        print(f"agent.targets={agent.targets}")
        print(f"agent.speed_counter.speed={agent.speed_counter.speed}")
        print(f"agent.speed_counter.max_speed={agent.speed_counter.max_speed}")
        print(f"agent.state={agent.state}")
        
    graph_env.step({0: RailEnvActions.MOVE_FORWARD})
    if graph_env.dones["__all__"]:
        print("==============================")
        print("done")
        break

## Switch geometry: which two legs stay closest?

For the switch node `SWITCH` (from the osmnx graph `GG` loaded back from `walensee.graphml`) we walk
`DISTANCE` metres along each of its three incident edges (starting at the switch) and compare the
resulting probe points pairwise: the pair that stays closest together is the one that continues
through the switch in (nearly) the same direction, the remaining leg is the odd one out.

In [ ]:
import itertools

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, Point

DISTANCE = 10  # metres along the edge, measured from the switch


def oriented_geometry(graph, u, v, k, start):
    """Geometry of edge (u, v, k) as a LineString that starts at node `start`."""
    geom = graph.edges[u, v, k].get("geometry")
    if geom is None:
        # unsimplified graph: the edge is the straight segment between its two nodes
        geom = LineString([(graph.nodes[u]["x"], graph.nodes[u]["y"]),
                           (graph.nodes[v]["x"], graph.nodes[v]["y"])])
    start_point = Point(graph.nodes[start]["x"], graph.nodes[start]["y"])
    if start_point.distance(Point(geom.coords[-1])) < start_point.distance(Point(geom.coords[0])):
        geom = geom.reverse()
    return geom


# GG was downcast to a plain DiGraph above (for the flatland env), but osmnx's
# projection/plotting helpers need a MultiDiGraph - rebuild one just for this section
GG_multi = nx.MultiDiGraph(GG)

# project to UTM so that lengths and interpolation are in metres
G_proj = ox.projection.project_graph(GG_multi)

# all edges incident to the switch, no matter their direction in the digraph
incident = ([(SWITCH, v, k, v) for _, v, k in G_proj.out_edges(SWITCH, keys=True)]
            + [(u, SWITCH, k, u) for u, _, k in G_proj.in_edges(SWITCH, keys=True)])

probes = {}
for u, v, k, neighbour in incident:
    geom = oriented_geometry(G_proj, u, v, k, SWITCH)
    if geom.length < DISTANCE:
        print(f"edge {SWITCH} -> {neighbour} is only {geom.length:.1f} m long, "
              f"probe clipped to its far end")
    probes[neighbour] = geom.interpolate(DISTANCE)

probe_points = gpd.GeoDataFrame({"neighbour": list(probes)},
                                geometry=list(probes.values()),
                                crs=G_proj.graph["crs"])
probe_points

In [ ]:
pairs = pd.DataFrame(
    [(a, b, pa.distance(pb)) for (a, pa), (b, pb) in itertools.combinations(probes.items(), 2)],
    columns=["neighbour_1", "neighbour_2", "distance"],
).sort_values("distance", ignore_index=True)
display(pairs)

closest = pairs.iloc[0]
print(f"at {DISTANCE} m from {SWITCH} the closest pair is "
      f"{closest.neighbour_1} / {closest.neighbour_2} ({closest.distance:.3f} m apart); "
      f"odd one out: {set(probes) - {closest.neighbour_1, closest.neighbour_2}}")

In [ ]:
# visual check: the switch (blue), its three incident edges (green), the three original
# neighbours (orange) and the three probe points at DISTANCE along each edge (red)
ego = nx.ego_graph(G_proj, SWITCH, radius=1, undirected=True)
ego_nodes, ego_edges = ox.convert.graph_to_gdfs(ego)
m_switch = ego_edges.to_crs(epsg=4326).explore(color="green")
# ego_nodes is indexed by osmid (not a column); explore() resets the index itself
# and surfaces it as an "osmid" tooltip column, so don't add that column again here
ego_nodes.loc[[SWITCH]].to_crs(epsg=4326).explore(m=m_switch, color="blue", tooltip="osmid", marker_kwds=mk)
ego_nodes.loc[list(probes)].to_crs(epsg=4326).explore(m=m_switch, color="orange", tooltip="osmid", marker_kwds=mk)
probe_points.to_crs(epsg=4326).explore(m=m_switch, color="red", tooltip="neighbour", marker_kwds=mk)

## Switch geometry as a reusable function

In [ ]:
def closest_neighbors(graph, switch, distance=DISTANCE):
    """Return the two neighbours of `switch` whose edges stay closest together.

    `graph` must be a (multi-)digraph projected to a metric CRS (see
    `ox.projection.project_graph`). We walk `distance` metres along each edge
    incident to `switch` and return the pair of neighbours whose probe points
    end up closest to each other - i.e. the leg pair that continues through
    the switch in (nearly) the same direction.
    """
    multi_graph = graph if graph.is_multigraph() else nx.MultiDiGraph(graph)

    incident = ([(switch, v, k, v) for _, v, k in multi_graph.out_edges(switch, keys=True)]
                + [(u, switch, k, u) for u, _, k in multi_graph.in_edges(switch, keys=True)])

    probes = {}
    for u, v, k, neighbour in incident:
        geom = oriented_geometry(multi_graph, u, v, k, switch)
        probes[neighbour] = geom.interpolate(min(distance, geom.length))

    pairs = pd.DataFrame(
        [(a, b, pa.distance(pb)) for (a, pa), (b, pb) in itertools.combinations(probes.items(), 2)],
        columns=["neighbour_1", "neighbour_2", "distance"],
    ).sort_values("distance", ignore_index=True)

    closest = pairs.iloc[0]
    return closest.neighbour_1, closest.neighbour_2


neighbor_1, neighbor_2 = closest_neighbors(G_proj, SWITCH)
print(f"closest neighbours of switch {SWITCH}: {neighbor_1} and {neighbor_2}")

## Sanity check: run `closest_neighbors` on every switch

In [ ]:
switches = [n for n in GG.nodes if GG.in_degree(n) + GG.out_degree(n) > 2]
print(f"found {len(switches)} node(s) with in+out degree > 2: {switches}")

missing_switch_attr = [n for n in switches if GG.nodes[n].get("railway") != "switch"]
assert not missing_switch_attr, (
    f"node(s) with in+out degree > 2 but no switch attribute: {missing_switch_attr}"
)
print("OK: all nodes with in+out degree > 2 have the switch attribute")

switch_attr_low_degree = [
    n for n, d in GG.nodes(data=True)
    if d.get("railway") == "switch" and GG.in_degree(n) + GG.out_degree(n) <= 2
]
print(f"node(s) with switch attribute but in+out degree <= 2: {switch_attr_low_degree}")

for switch in switches:
    try:
        n1, n2 = closest_neighbors(G_proj, switch)
        print(f"switch {switch}: closest neighbours are {n1} and {n2}")
    except Exception as e:
        print(f"switch {switch}: closest_neighbors FAILED with {type(e).__name__}: {e}")

## Visual inspection: switches vs. low-degree switch-tagged nodes

Plot the whole graph, highlighting nodes with in+out degree > 2 (the switches used above) in one
color and nodes that carry the `switch` attribute but have degree <= 2 (flagged by the sanity
check above) in another.

In [ ]:
from matplotlib.lines import Line2D

switches_set = set(switches)
switch_attr_low_degree_set = set(switch_attr_low_degree)

default_color = "#999999"
switch_color = "#1f77b4"
low_degree_switch_color = "#d62728"

node_color = [
    low_degree_switch_color if n in switch_attr_low_degree_set
    else switch_color if n in switches_set
    else default_color
    for n in GG_multi.nodes
]

fig, ax = ox.plot.plot_graph(
    GG_multi,
    node_color=node_color,
    node_size=15,
    edge_color="w",
    edge_linewidth=0.5,
    show=False,
    close=False,
)
ax.legend(
    handles=[
        Line2D([], [], marker="o", linestyle="", color=switch_color, label="switch (in+out degree > 2)"),
        Line2D([], [], marker="o", linestyle="", color=low_degree_switch_color, label="switch attribute, in+out degree <= 2"),
        Line2D([], [], marker="o", linestyle="", color=default_color, label="other node"),
    ],
    loc="lower left",
)

In [ ]:
# same categories as above, but as an interactive map (see "visual check" above)
nodes_all, edges_all = ox.convert.graph_to_gdfs(GG_multi)

other_nodes = nodes_all.index.difference(switches_set | switch_attr_low_degree_set)

m_all = edges_all.explore(color="green")
nodes_all.loc[other_nodes].explore(m=m_all, color=default_color, tooltip="osmid", marker_kwds=mk)
nodes_all.loc[sorted(switches_set)].explore(m=m_all, color=switch_color, tooltip="osmid", marker_kwds=mk)
nodes_all.loc[sorted(switch_attr_low_degree_set)].explore(m=m_all, color=low_degree_switch_color, tooltip="osmid", marker_kwds=mk)